<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/resnet_split_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시작

In [14]:
import os
import torch
import torch.nn as nn
import torch.distributed as dist

from torchvision import models
from torch.distributed.pipelining import pipeline, SplitPoint


# -------------------------
# dist init (single process)
# -------------------------
def init_dist():
    if dist.is_initialized():
        print("Distributed process group already initialized.")
        return

    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29500"

    dist.init_process_group(
        backend="nccl",
        rank=0,
        world_size=1,
    )
    torch.cuda.set_device(0)


def cleanup_dist():
    if dist.is_initialized():
        dist.destroy_process_group()


# -------------------------
# ResNet split helpers
# -------------------------
def ordered_split_candidates(resnet: nn.Module):
    names = []
    for li in range(1, 5):
        layer = getattr(resnet, f"layer{li}")
        for bi in range(len(layer)):
            names.append(f"layer{li}.{bi}")
    return names


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def build_split_spec(resnet: nn.Module):
    """
    단일 GPU 검증용:
    - stage 수는 2로 가정
    - 실제 병렬 실행 목적 아님
    """
    candidates = ordered_split_candidates(resnet)

    total = sum(count_params(resnet.get_submodule(n)) for n in candidates)
    acc = 0
    split_name = None

    for name in candidates:
        acc += count_params(resnet.get_submodule(name))
        if acc >= total / 2:
            split_name = name
            break

    assert split_name is not None
    return {split_name: SplitPoint.BEGINNING}


# -------------------------
# main
# -------------------------
def main():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU 필요")

    # Ensure cleanup is called before init if the main function is re-run within the same kernel session
    cleanup_dist()

    init_dist()
    device = torch.device("cuda:0")

    # 1) 모델 (resnet 50 / 101 / 152)
    model = models.resnet101(weights=None).eval()

    # 2) split_spec 생성
    split_spec = build_split_spec(model)
    print("split_spec:", split_spec)

    # 3) pipeline 생성 (tracing용 micro-batch)
    example_mb = torch.randn(1, 3, 224, 224)

    pipe = pipeline(
        module=model,
        mb_args=(example_mb,),
        split_spec=split_spec,
    )

    print("✅ pipeline() 생성 성공")
    print("pipe.num_stages =", pipe.num_stages)
    for i in range(pipe.num_stages):
      stage_module = pipe.get_stage_module(i)
      print(f'stage{i} modules:')
      for name, module in stage_module.named_children():
        print(f'  - {name}: {module.__class__.__name__}')

    # 4) stage 생성 확인
    stage0 = pipe.build_stage(
        stage_index=0, # Changed from stage_idx to stage_index
        device=device,
        group=dist.group.WORLD,  # dist init 했으므로 WORLD 사용
    )

    print("✅ build_stage(stage_index=0) 성공")
    print("stage type:", type(stage0))

    cleanup_dist()


if __name__ == "__main__":
    main()

split_spec: {'layer3.17': <SplitPoint.BEGINNING: 1>}
✅ pipeline() 생성 성공
pipe.num_stages = 2
stage0 modules:
  - conv1: InterpreterModule
  - bn1: InterpreterModule
  - relu: InterpreterModule
  - maxpool: InterpreterModule
  - layer1: InterpreterModule
  - layer2: InterpreterModule
  - layer3: InterpreterModule
stage1 modules:
  - layer3: InterpreterModule
  - layer4: InterpreterModule
  - avgpool: InterpreterModule
  - fc: InterpreterModule
✅ build_stage(stage_index=0) 성공
stage type: <class 'torch.distributed.pipelining.stage._PipelineStage'>


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
